In [ ]:
# Import Required Libraries
import os

DATASETS_ROOT = "dataSets"

if __name__ == "__main__":
    if not os.path.isdir(DATASETS_ROOT):
        print(f"Datasets root directory '{DATASETS_ROOT}' not found.")
    else:
        for dataset_name in os.listdir(DATASETS_ROOT):
            dataset_path = os.path.join(DATASETS_ROOT, dataset_name)
            if os.path.isdir(dataset_path):
                print(f"Processing dataset: {dataset_name}") #testing dataset name

Processing dataset: indianexpress
Processing dataset: universityherald
Processing dataset: kotiliesi
Processing dataset: ruoka_fi
Processing dataset: theguardian


In [ ]:
# Import Required Libraries
import os
from bs4 import BeautifulSoup
from collections import Counter

DATASETS_ROOT = "dataSets"
TARGET_TAGS = ["title", "h1", "h2", "h3", "h4", "p", "a", "strong", "b", "em"]


def analyze_local_dataset(dataset_dir):
    tag_counts = Counter()
    total_keywords_found = 0

    # Recursively find .htm/.html files in all subdirectories
    html_files = []
    for root, dirs, files in os.walk(dataset_dir):
        for file in files:
            if file.endswith(".htm") or file.endswith(".html"):
                html_files.append(os.path.join(root, file))

    print(f"Analyzing {len(html_files)} files in {dataset_dir}...")

    for file_path in html_files:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            soup = BeautifulSoup(f.read(), "lxml")

        # 1. Extract Ground Truth from Meta Tags
        gt_keywords = []
        # Extract from <meta name="keywords"> and <meta name="news_keywords">
        meta_tags = soup.find_all("meta", attrs={"name": True})
        for meta in meta_tags:
            name = meta.get("name", "").lower()
            if name in ["keywords", "news_keywords"] and meta.get("content"):
                gt_keywords += [k.strip().lower() for k in meta["content"].split(",")]

        # Extract from <meta property="article:tag" content="...">
        property_tags = soup.find_all("meta", attrs={"property": True})
        for meta in property_tags:
            prop = meta.get("property", "").lower()
            if prop == "article:tag" and meta.get("content"):
                gt_keywords.append(meta["content"].strip().lower())

        # Remove duplicates
        gt_keywords = list(set(gt_keywords))
        if not gt_keywords:
            continue

        # 2. Analyze occurrences in HTML elements
        for tag_name in TARGET_TAGS:
            elements = soup.find_all(tag_name)
            for el in elements:
                text = el.get_text().lower()
                for word in gt_keywords:
                    # Simple check if the keyword appears in the tag text
                    if word in text:
                        tag_counts[tag_name] += 1
                        total_keywords_found += 1

    return tag_counts, total_keywords_found


if __name__ == "__main__":
    if not os.path.isdir(DATASETS_ROOT):
        print(f"Datasets root directory '{DATASETS_ROOT}' not found.")
    else:
        for dataset_name in os.listdir(DATASETS_ROOT):
            dataset_path = os.path.join(DATASETS_ROOT, dataset_name)
            if os.path.isdir(dataset_path):
                stats, total_found = analyze_local_dataset(dataset_path)

Analyzing 658 files in dataSets/indianexpress...
